<a href="https://colab.research.google.com/github/hadiah115-tech/Urdu-OCR_Project/blob/main/SI26_Week4_Hadia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install transformers==4.41.2 tokenizers==0.19.1 sentencepiece torch pillow pandas

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
import os

In [27]:
!git clone https://github.com/hadiah115-tech/Urdu-OCR_Project.git

Cloning into 'Urdu-OCR_Project'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 924 (delta 259), reused 11 (delta 11), pack-reused 658 (from 1)
Receiving objects: 100% (924/924), 1.65 MiB | 8.54 MiB/s, done.
Resolving deltas: 100% (283/283), done.


In [28]:
import os
os.chdir("/content/Urdu-OCR_Project")
print(os.getcwd())

/content/Urdu-OCR_Project


In [29]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
print("Processor Loaded Successfully!")

Processor Loaded Successfully!


In [30]:
class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row["image"]).convert("RGB")

        # Process image
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process text label
        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [31]:
dataset = UrduOCRDataset("data/labels.csv", processor)

Dataset loaded: 200 samples


In [32]:
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)

print("Dataset is working correctly!")

Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!


In [33]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")

Training samples: 160
Testing samples: 40


In [34]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

print("Train DataLoader created successfully!")
print("Test DataLoader created successfully!")

Train DataLoader created successfully!
Test DataLoader created successfully!


In [35]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [36]:
!pip install transformers

In [37]:
from transformers import TrOCRProcessor
from transformers import VisionEncoderDecoderModel

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model Loaded Successfully")

cuda


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model Loaded Successfully


In [38]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4
)

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print(len(train_loader))

40


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [39]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    print(f"\nEpoch {epoch+1}")

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)

        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:

            print(batch_idx, loss.item())

    avg_loss = total_loss / len(train_loader)

    print("Average Loss =", avg_loss)

print("Training Complete")


Epoch 1
0 17.577442169189453
10 3.968259811401367
20 2.8468751907348633
30 2.1765429973602295
Average Loss = 3.9933717787265777

Epoch 2
0 2.5250844955444336
10 2.722869396209717
20 2.5731821060180664
30 2.4062581062316895
Average Loss = 2.514766889810562

Epoch 3
0 2.5450592041015625
10 2.5143320560455322
20 2.387819766998291
30 2.606656789779663
Average Loss = 2.4845448672771453
Training Complete


In [40]:

model.eval()

model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 128
model.config.num_beams = 4
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 2
model.config.length_penalty = 2.0

print("=== Model Evaluation on Test Images ===")
print()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].clone()

        # Replace -100 with PAD token
        labels[labels == -100] = processor.tokenizer.pad_token_id

        generated_ids = model.generate(
            pixel_values,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Predicted:", pred)
            print("Actual   :", actual)
            print("-" * 50)

accuracy = (correct / total) * 100 if total > 0 else 0

print()
print("=" * 40)
print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")
print("=" * 40)

=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1283: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Predicted: ��ا �ر �و��ْنا������ا� ��ر� ی�و۩ہ��ے�ن�ا������ا� ��ر� ڌ�وکځ��ڒ�ن�ا����اا �ارا ا�اوا�ا�ا�ا�اناا�ا�ا � �ر �  ،
Actual   : و امان کے قیام میں سیکورٹی اداروں کی ناکامی، حکومت
--------------------------------------------------
Predicted: ��ا �ر� �و�ْنا��ت����ا� ��ری� ۩�وہ��ے�ن�ا����ت���ا� ��رڌ� ک�وځ��ڒ�ن�ا����ت�اا �ارا�ا ا�اوا�ا�ا�اناا�ا�اتا
Actual   : نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظاہرین نے
--------------------------------------------------
Predicted: ��ا �ر� �و���ن�ا�����ا� ��ری� ۩�وہے���ن���ا����ا� ��رڌ� ک�وځڒ���ن���ا��اا �ارا�ا ا�اوا�ا�ا�انا�اا�ا � �ر ، � 
Actual   : کے بعد ان کا سعودی عرب جانا ضروری تھا اس
--------------------------------------------------
Predicted: ��ا �ر �و��ْنا������ا� ��ر� ی�و۩ہ��ے�ن�ا������ا� ��ر� ڌ�وکځ��ڒ�ن�ا����اا �ارا ا�اوا�ا�ا�ا�اناا�ا�ا � �ر �  ،
Actual   : بیان ڈی جی حج آپریشن راؤ شکیل احمد کا نام
--------------------------------------------------
Predicted: �� �ار� �و�ْنا������ ��ا�ری� ۩�وہ��ے�ن�ا������ ��ا�رڌ� ک�وځ��ڒ�ن�ا���� � �

In [41]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [42]:
print(model.config.decoder_start_token_id)
print(model.config.pad_token_id)
print(model.config.eos_token_id)

0
1
2


In [43]:
batch = next(iter(test_loader))

print(batch["labels"][0][:30])

tensor([    0, 33866,  1437, 29438, 33070, 29438, 31746,  1437, 46164, 15375,
        44148, 10659, 36963,  9264, 44148, 14285, 29438, 33070, 42620, 44148,
        14285, 46164,  3070, 25790, 15264, 44148, 14285, 46164, 15375, 33866])


In [44]:
generated_ids = model.generate(batch["pixel_values"][:1].to(device))

print("Prediction:")
print(processor.batch_decode(generated_ids, skip_special_tokens=True))

labels = batch["labels"][0].clone()
labels[labels == -100] = processor.tokenizer.pad_token_id

print("Actual:")
print(processor.batch_decode(labels.unsqueeze(0), skip_special_tokens=True))

Prediction:
['��ا �ر �و��ْنا������ا� ��ر� ی�و۩ہ��ے�ن�ا������ا� ��ر� ڌ�وکځ��ڒ�ن�ا����اا �ارا ا�اوا�ا�ا�ا�اناا�ا�ا � �ر �  ،']
Actual:
['و امان کے قیام میں سیکورٹی اداروں کی ناکامی، حکومت']


In [45]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
save_path = "/content/drive/MyDrive/SI26-urdu-ocr-model"

model.save_pretrained(save_path)

processor.save_pretrained(save_path)

print("Model Saved")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 128, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 2}


Model Saved
